In [1]:
# !pip install langchain_huggingface
# !pip install sentence-transformers

## load jsonl file

In [ ]:
from __future__ import annotations

from typing import List, Dict, Any, Optional
import os
import asyncio
import orjson
from tqdm import tqdm

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from src.faiss_storage import FaissStorage


EMB_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMB_DIM = 384
CHUNK_SIZE=512
CHUNK_OVERLAP=200

embed_model = HuggingFaceEmbeddings(model_name=EMB_MODEL_NAME)
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)


# load the pdf file from the path and chunk it
def stream_jsonl(path: str) -> Iterator[Dict[str, Any]]:
    with open(path, "rb") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                yield orjson.loads(line)
            except orjson.JSONDecodeError as e:
                raise ValueError(f"Bad JSON on line {line_no}: {e}") from e

def chunk_text(
    text: str,
    chunk_size: int = 900,
    overlap: int = 150,
    min_chars: int = 200,
):
    text = text.strip()
    if not text:
        return []

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", " ", ""],  # paragraph → line → word → char
        length_function=len,
    )

    chunks = splitter.split_text(text)

    # Optional: enforce minimum length (like your current logic)
    chunks = [c.strip() for c in chunks if len(c.strip()) >= min_chars]

    return chunks

def build_payloads(rec: Dict[str, Any], chunks: List[str]) -> List[Dict[str, Any]]:
    doc_id = str(rec.get("doc_id", ""))
    title = str(rec.get("title", ""))
    collection = str(rec.get("collection", ""))
    source_dir = str(rec.get("source_dir", ""))
    document_type = str(rec.get("document_type", ""))

    payloads = []
    for i, ch in enumerate(chunks):
        payloads.append({
            "text": ch,
            "source": source_dir or doc_id,     # what you want to show as citation/source
            "doc_id": doc_id,
            "chunk_id": i,
            "title": title,
            "document_type": document_type,
            "collection": collection,
            "date": rec.get("date", ""),
            "date_numeric": rec.get("date_numeric", None),
            "ocr_quality": rec.get("ocr_quality", None),
            "ocr_quality_tier": rec.get("ocr_quality_tier", None),
        })
    return payloads



# def embed_texts(texts: list[str]) -> list[list[float]]:
#     response = embed_model.embed_documents(texts)
#     return response

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [1]:
from src.custom_types import RAGChunkAndSrc, RAGUpsertResult, RAGSearchResult, RAQQueryResult
import uuid
import os
from src.ingest_corpus_jsonl import load_and_chunk_jsonl, embed_texts
from src.faiss_storage import FaissStorage


path_to_corpus = "data/corpus2.jsonl"

def _process_corpus(jsonl_path) -> RAGUpsertResult:
    # jsonl_path = ctx.event.data["jsonl_path"]
    # source_id = ctx.event.data.get("source_id", jsonl_path)

    source_id = jsonl_path

    # Stream records, clean, chunk per record (keep metadata!)
    chunks = load_and_chunk_jsonl(jsonl_path)
    
    texts = [c["text"] for c in chunks]
    vecs = embed_texts(texts)
    ids = [str(uuid.uuid5(uuid.NAMESPACE_URL, f"{source_id}:{i}")) for i in range(len(chunks))]
    
    payloads = []
    for i, c in enumerate(chunks):
        payload = {"source": source_id, "text": c["text"]}
        payload.update(c.get("metadata", {}))
        payloads.append(payload)
        
    FaissStorage().upsert(ids, vecs, payloads)
    return RAGUpsertResult(ingested=len(chunks))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
# results = _process_corpus(path_to_corpus)

question = "Who was gandhi ?"
top_k = 5
query_vec = embed_texts([question])[0]  # 1. convert the question into vector
store = FaissStorage()
found = store.search(query_vec, top_k)

In [ ]:
from langchain_community.retrievers import BM25Retriever
from src.ingest_corpus_jsonl import load_and_chunk_jsonl, embed_texts

path_to_corpus = "data/corpus2.jsonl"
chunks = load_and_chunk_jsonl(jsonl_path=path_to_corpus)
texts = [c["text"] for c in chunks]

all_metadata = [chunk.get("metadata", {"source": "unkown"}) for chunk in chunks]

print(chunks)

# Initialize the BM25 retriever with texts and metadatas
# bm25_retriever = BM25Retriever.from_texts(texts, metadatas=metadatas)
# bm25_retriever.k = 5

# # Run a query against the retriever
# query = "your search query here"
# results = bm25_retriever.invoke(query)
# results

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
from src.retrievers import DenseRetriever
from src.faiss_storage import FaissStorage

question = "who was mahatma gandhi?"
retriever = DenseRetriever(faiss_store=FaissStorage())
found = retriever.search(question, top_k=10)
print(found)


{'contexts': ["in June Feb. â\x80\x93 Budget in\n1 March\nWe returned to Ranchi in May.\n14.\n1.\nThe has cooperation of praise silentioned\nigtengine\nNow must in ticket, whee\nand\nHose had been attacks in Europeans, Madis\niflucci nations rest have still\nGreat influence and be masses\nThe how cooperators were still active in all\ntestions plus fourrice. The empaign had been\nmost intensive in Tithat where Gandhi's\nreputation is a saint apprelia most stradley\n1s Mr Masses ( real of his formines advanian\napation against his plantors and tire had\nBeen attacks a Europeans a Officers.\nGandhi's visits One and stated\ninterest thers. The lead was taken by parations\nPopite the bass. Casses of was reused\nat Students at Ravenshaw College\nShade bige School. Apirators war Marti\ntLC.V.\nfget at he Anissional bribes in Caste Moor\nfrittender\nuim for his\no date sense Mr May there was a serious\nwit a Mini\nwon the", 'put some of my ideas with brief dikett directness, even bru-\ntally i

In [7]:
found['contexts']

["in June Feb. â\x80\x93 Budget in\n1 March\nWe returned to Ranchi in May.\n14.\n1.\nThe has cooperation of praise silentioned\nigtengine\nNow must in ticket, whee\nand\nHose had been attacks in Europeans, Madis\niflucci nations rest have still\nGreat influence and be masses\nThe how cooperators were still active in all\ntestions plus fourrice. The empaign had been\nmost intensive in Tithat where Gandhi's\nreputation is a saint apprelia most stradley\n1s Mr Masses ( real of his formines advanian\napation against his plantors and tire had\nBeen attacks a Europeans a Officers.\nGandhi's visits One and stated\ninterest thers. The lead was taken by parations\nPopite the bass. Casses of was reused\nat Students at Ravenshaw College\nShade bige School. Apirators war Marti\ntLC.V.\nfget at he Anissional bribes in Caste Moor\nfrittender\nuim for his\no date sense Mr May there was a serious\nwit a Mini\nwon the",
 'put some of my ideas with brief dikett directness, even bru-\ntally it may seam.\

In [11]:
from src.retrievers import DenseRetriever, SparseRetriever
from src.faiss_storage import FaissStorage

question = "who were salves in india?"
retriever = SparseRetriever(faiss_store=FaissStorage())
found = retriever.search(question, top_k=10)
print(found)


{'contexts': ['of him was by a but the fact is so.\nVessel that had stopped\nIn examining the question, which\nat St Helerra some\nmonths back and by\nwhich time 1 are the best Stations for men of war, to cruige\nAdmiral must havion in India? There are two Points of\nbeen in England', '176\nto add to the row in the trouse the\nWhad is perpetual barfure with\nthe contractors workmen, I hear tim\nThermishing from room to sour, now\nasputtering fire of Musquetuitry in\nthe lower regions thin it rolls up\nstairs a valley firing commences\nat regular internals of half on\nnour there is a general action, and\nalt his ling guns come into plays.\nfiring salves of the strongest lan\nquage in Maharatter, tilt I wonder\nthey do not blow the root ofte\nI lorgot to tell you I saw sommy\nBelt in Bombay & sold his a horse\nus I can hardly apporch to keep one\nnow a days. He introduced me to\nhis crise, she is pleasing toothing her\nno teality hardly pretty, I will soon\nfade under un castern sum. He 

In [12]:
found["contexts"]

['of him was by a but the fact is so.\nVessel that had stopped\nIn examining the question, which\nat St Helerra some\nmonths back and by\nwhich time 1 are the best Stations for men of war, to cruige\nAdmiral must havion in India? There are two Points of\nbeen in England',
 '176\nto add to the row in the trouse the\nWhad is perpetual barfure with\nthe contractors workmen, I hear tim\nThermishing from room to sour, now\nasputtering fire of Musquetuitry in\nthe lower regions thin it rolls up\nstairs a valley firing commences\nat regular internals of half on\nnour there is a general action, and\nalt his ling guns come into plays.\nfiring salves of the strongest lan\nquage in Maharatter, tilt I wonder\nthey do not blow the root ofte\nI lorgot to tell you I saw sommy\nBelt in Bombay & sold his a horse\nus I can hardly apporch to keep one\nnow a days. He introduced me to\nhis crise, she is pleasing toothing her\nno teality hardly pretty, I will soon\nfade under un castern sum. He is\ntrying t